## PyINE prompt result database repackaging for Hugging Face

This notebook loads prompt result records from the framework's SQLite database and pushes them to a
HuggingFace dataset repository in Parquet format. Each row corresponds to a single LLM prompting
result, with structured columns for filtering and JSON columns for complex metadata fields.

**Prerequisites:** `pip install datasets huggingface_hub` and `huggingface-cli login`.

In [ ]:
import collections.abc
import json
import re

import datasets
import huggingface_hub
import tqdm

import pyine.data.traces.dataset_utils
import pyine.data.utils.splits
import pyine.prompts
import pyine.utils.filesystem
import pyine.utils.reprod

pyine.utils.reprod.entrypoint_setup()

In [ ]:
# ------------ SETTINGS ------------
SOURCE_DATASET_NAME = "TACO"
HF_REPO_NAME = "plstcharles-saifh/pyine-v1-augments"
SOURCE_DATASET_REPO = "plstcharles-saifh/pyine-v1-traces"
MAX_SHARD_SIZE = "1GB"

# optional filters (set to None to include everything)
TARGET_PROMPT_NAMES: list[str] | None = None  # e.g. ["hints/docs", "issues/todos"]
TARGET_PROMPT_VERSION: str | None = None
TAG_FILTER_RULE: str | None = None
# ----------------------------------

### Load prompt result database

In [ ]:
result_db = pyine.prompts.get_framework_db()
print(f"database path: {pyine.prompts.get_framework_db_path()}")
print(f"total records: {result_db.count_entries():,}")
print(f"prompt names: {result_db.list_prompt_names()}")
print(f"unique groups: {len(result_db.list_groups()):,}")
print(f"unique identifiers: {len(result_db.list_identifiers()):,}")

In [ ]:
# load records (optionally filtered by prompt name)
if TARGET_PROMPT_NAMES is not None:
    all_records: list[pyine.prompts.PromptResultRecord] = []
    for prompt_name in TARGET_PROMPT_NAMES:
        all_records.extend(
            result_db.get_by_prompt_name(
                prompt_name=prompt_name,
                prompt_version=TARGET_PROMPT_VERSION,
                tag_filter_rule=TAG_FILTER_RULE,
            )
        )
else:
    all_records = result_db.get_all_results(tag_filter_rule=TAG_FILTER_RULE)

print(f"loaded {len(all_records):,} records for export")

In [ ]:
# load the pre-computed dataset split to map problems to pyine subsets
split_result = pyine.data.utils.splits.get_dataset_split_result(SOURCE_DATASET_NAME)
subset_assignments = split_result.subset_assignments
print(f"split subsets: {split_result.config.subset_names}")
print(f"split covers {len(subset_assignments):,} problems")

# matches the problem key prefix (e.g. "TACO/train/p000001") at the start of any identifier,
# including record_uid-style identifiers that have _promptname_timestamp_hash suffixes appended
# by the annotation app (see _build_record_uid_prefix in pyine.prompts.result_db)
_PROBLEM_KEY_PATTERN = re.compile(r"^([^/]+/[^/]+/p\d+)")


def _get_problem_key(identifier: str) -> str | None:
    """Extracts the problem identifier string from any identifier format.

    Tries clean parsing first (trace -> solution -> problem hierarchy), then falls back
    to regex prefix extraction for record_uid-style identifiers used by the annotation app.
    """
    # trace level: "TACO/train/p000001/s0000/t0001" or with "/a:..." suffix
    try:
        trace_id = pyine.data.traces.dataset_utils.TraceIdentifier.from_string(identifier)
        return repr(trace_id.get_parent_identifier().get_parent_identifier())
    except ValueError:
        pass
    # solution level: "TACO/train/p000001/s0001"
    try:
        sol_id = pyine.data.traces.dataset_utils.SolutionIdentifier.from_string(identifier)
        return repr(sol_id.get_parent_identifier())
    except ValueError:
        pass
    # problem level: "TACO/train/p000001"
    try:
        prob_id = pyine.data.traces.dataset_utils.CodingProblemIdentifier.from_string(identifier)
        return repr(prob_id)
    except ValueError:
        pass
    # fallback: extract the problem key prefix from record_uid-style identifiers
    # (e.g. "TACO/train/p000001/s0001/t0056_issues/docs_20260208-025232_e4dd44")
    match = _PROBLEM_KEY_PATTERN.match(identifier)
    if match:
        try:
            return repr(pyine.data.traces.dataset_utils.CodingProblemIdentifier.from_string(match.group(1)))
        except ValueError:
            pass
    return None


# build a lookup from record identifier to pyine subset
unresolved_ids: list[str] = []
identifier_to_subset: dict[str, str | None] = {}
for record in all_records:
    if record.identifier not in identifier_to_subset:
        problem_key = _get_problem_key(record.identifier)
        subset = subset_assignments.get(problem_key) if problem_key else None
        identifier_to_subset[record.identifier] = subset
        if subset is None:
            unresolved_ids.append(record.identifier)

subset_dist = collections.Counter(identifier_to_subset.values())
print(f"\nsubset assignment for {len(identifier_to_subset):,} unique identifiers:")
for subset_name in split_result.config.subset_names:
    print(f"  {subset_name}: {subset_dist.get(subset_name, 0):,}")
if unresolved_ids:
    print(f"  unresolved: {len(unresolved_ids):,}")
    print(f"\nfirst {min(50, len(unresolved_ids))} unresolved identifiers:")
    for unresolved_id in unresolved_ids[:50]:
        problem_key = _get_problem_key(unresolved_id)
        print(f"  {unresolved_id}  (problem_key={problem_key})")

### Define record-to-row conversion

Each `PromptResultRecord` is flattened with:
- **Structured columns** for identity, prompt name/version, group, tags, creation metadata
- **Full text columns** for the prompt and LLM response
- **JSON string columns** for complex nested metadata (`meta`, `llm_params`, `llm_output`)

In [ ]:
def _safe_json_dumps(value: object) -> str:
    """Serializes a value to a JSON string, falling back to repr for non-serializable types."""
    if value is None:
        return "null"
    try:
        return json.dumps(value, ensure_ascii=False)
    except (TypeError, ValueError):
        return json.dumps(repr(value), ensure_ascii=False)


def record_to_hf_row(
    record: pyine.prompts.PromptResultRecord,
    pyine_subset: str | None,
) -> dict:
    """Converts a PromptResultRecord into a flat dict suitable for HF datasets."""
    cmeta = record.creation_meta
    return {
        # ---- identity ----
        "record_uid": record.record_uid,
        "identifier": record.identifier,
        "prompt_name": record.prompt_name,
        "prompt_version": record.prompt_version,
        "group": record.group,
        "tags": record.tags,
        "pyine_subset": pyine_subset,
        # ---- prompt + response (full text) ----
        "prompt": record.prompt,
        "result": record.result,
        # ---- creation metadata (flattened) ----
        "created_at": cmeta.created_at.isoformat(),
        "created_by": cmeta.created_by,
        "platform": cmeta.platform,
        "provider": cmeta.provider,
        # ---- complex metadata (JSON-serialized) ----
        "meta_json": _safe_json_dumps(record.meta),
        "llm_params_json": _safe_json_dumps(cmeta.llm_params),
        "llm_output_json": _safe_json_dumps(cmeta.llm_output),
    }

### Build HF dataset via generator

In [ ]:
def prompt_results_generator() -> collections.abc.Iterator[dict]:
    """Yields one HF-compatible dict per prompt result record."""
    for record in tqdm.tqdm(all_records, desc="repackaging prompt results"):
        pyine_subset = identifier_to_subset.get(record.identifier)
        yield record_to_hf_row(record, pyine_subset=pyine_subset)


hf_cache_dir = pyine.utils.filesystem.get_data_cache_subdir("hf_prompt_rpkg")
hf_dataset = datasets.Dataset.from_generator(
    prompt_results_generator,
    cache_dir=str(hf_cache_dir),
)
print(f"created HF dataset with {len(hf_dataset):,} rows and {len(hf_dataset.column_names)} columns")
print(f"columns: {hf_dataset.column_names}")

# split into a DatasetDict by pyine_subset so HF shows proper train/valid/test splits
hf_dataset_dict = datasets.DatasetDict()
for subset_name in split_result.config.subset_names:
    subset_ds = hf_dataset.filter(lambda row, sn=subset_name: row["pyine_subset"] == sn)
    hf_dataset_dict[subset_name] = subset_ds
    print(f"  {subset_name}: {len(subset_ds):,} rows")

# include any unresolved records in a separate split (if any exist)
unresolved_ds = hf_dataset.filter(lambda row: row["pyine_subset"] is None)
if len(unresolved_ds) > 0:
    hf_dataset_dict["unassigned"] = unresolved_ds
    print(f"  unassigned: {len(unresolved_ds):,} rows")

### Preview a sample row

In [ ]:
first_split = split_result.config.subset_names[0]
sample = hf_dataset_dict[first_split][0]
print(f"sample from '{first_split}' split:")
for key, value in sample.items():
    if isinstance(value, str) and len(value) > 200:
        print(f"  {key}: {value[:200]}... ({len(value)} chars)")
    else:
        print(f"  {key}: {value}")

### Build dataset card and push to Hugging Face Hub

In [ ]:
# attach database-level summary metadata to the HF dataset info
prompt_names = result_db.list_prompt_names()
prompt_name_counts = result_db.count_entries(prompt_name=prompt_names, breakdown=True)
dataset_level_metadata = {
    "source_dataset_repo": SOURCE_DATASET_REPO,
    "source_db_path": str(pyine.prompts.get_framework_db_path()),
    "total_records": len(all_records),
    "prompt_names": prompt_names,
    "prompt_name_counts": {name_tuple[0]: count for name_tuple, count in prompt_name_counts.items()},
    "unique_groups": len(result_db.list_groups()),
    "unique_identifiers": len(result_db.list_identifiers()),
    "pyine_subset_names": split_result.config.subset_names,
}
if TARGET_PROMPT_NAMES is not None:
    dataset_level_metadata["filter_prompt_names"] = TARGET_PROMPT_NAMES
if TARGET_PROMPT_VERSION is not None:
    dataset_level_metadata["filter_prompt_version"] = TARGET_PROMPT_VERSION
if TAG_FILTER_RULE is not None:
    dataset_level_metadata["filter_tag_rule"] = TAG_FILTER_RULE

for split_ds in hf_dataset_dict.values():
    split_ds.info.description = (
        f"PyINE-v1 LLM prompted code augmentation result database export. "
        f"Contains {len(all_records):,} records across {len(prompt_names)} prompt types."
    )
    split_ds.info.dataset_name = HF_REPO_NAME.split("/")[-1]

print(f"dataset-level metadata: {json.dumps(dataset_level_metadata, indent=2, default=str)}")

In [ ]:
# build per-prompt-name breakdown for the card
prompt_name_counts_map = {name_tuple[0]: count for name_tuple, count in prompt_name_counts.items()}
prompt_table_rows = "\n".join(
    f"| `{name}` | {prompt_name_counts_map.get(name, 0):,} |" for name in sorted(prompt_name_counts_map.keys())
)

# collect unique providers and platforms from the records
providers = sorted({r.creation_meta.provider for r in all_records if r.creation_meta.provider})
models = sorted(
    {
        r.creation_meta.llm_params.get("model_kwargs", {}).get("model", "?")
        for r in all_records
        if r.creation_meta.llm_params
    }
)

filter_note = ""
if TARGET_PROMPT_NAMES or TARGET_PROMPT_VERSION or TAG_FILTER_RULE:
    filter_parts = []
    if TARGET_PROMPT_NAMES:
        filter_parts.append(f"prompt names: `{TARGET_PROMPT_NAMES}`")
    if TARGET_PROMPT_VERSION:
        filter_parts.append(f"prompt version: `{TARGET_PROMPT_VERSION}`")
    if TAG_FILTER_RULE:
        filter_parts.append(f"tag filter: `{TAG_FILTER_RULE}`")
    filter_note = (
        "\n> **Note:** This export was filtered to a subset of the full database: " + ", ".join(filter_parts) + ".\n"
    )


def _get_hf_size_category(count: int) -> str:
    """Returns the HuggingFace size category string for a given item count."""
    if count >= 1_000_000_000_000:
        return "n>1T"
    if count >= 1_000_000_000:
        return "1B<n<1T"
    if count >= 100_000_000:
        return "100M<n<1B"
    if count >= 10_000_000:
        return "10M<n<100M"
    if count >= 1_000_000:
        return "1M<n<10M"
    if count >= 100_000:
        return "100K<n<1M"
    if count >= 10_000:
        return "10K<n<100K"
    if count >= 1_000:
        return "1K<n<10K"
    return "n<1K"


card_data = huggingface_hub.DatasetCardData(
    language="en",
    license="cc-by-4.0",
    source_datasets=[SOURCE_DATASET_REPO],
    tags=["code", "code-augmentation", "llm-annotations", "python", "code-analysis", "prompt-results"],
    task_categories=["text-generation"],
    size_categories=[_get_hf_size_category(len(all_records))],
    pretty_name="PyINE-v1 LLM Prompted Code Augmentations",
)

card_content = f"""\
# PyINE-v1 Prompted Code Augmentations

This dataset contains **{len(all_records):,}** LLM prompting results generated by the
[PyINE](https://github.com/saifh-github/pyine) framework. Each row is a single prompt/response
pair produced by querying an LLM to annotate or augment Python code solutions (e.g.
adding documentation hints, injecting bugs, generating misleading comments).
{filter_note}
## Dataset structure

### Prompt types

| Prompt name | Records |
|-------------|---------|
{prompt_table_rows}

### Columns

**Identity:**
| Column | Type | Description |
|--------|------|-------------|
| `record_uid` | `string` | Unique record ID (content-hashed) |
| `identifier` | `string` | Record identifier (typically a trace or solution ID) |
| `prompt_name` | `string?` | Name of the prompt template used |
| `prompt_version` | `string?` | Version of the prompt template |
| `group` | `string?` | Optional grouping label |
| `tags` | `list[string]` | Tags for filtering (e.g. `augment:*`, `created_by:*`) |

**Prompt and response:**
| Column | Type | Description |
|--------|------|-------------|
| `prompt` | `string` | Full prompt text sent to the LLM |
| `result` | `string` | Full LLM response text |

**Creation metadata:**
| Column | Type | Description |
|--------|------|-------------|
| `created_at` | `string` | ISO 8601 UTC timestamp |
| `created_by` | `string` | Username/identifier of the creator |
| `platform` | `string` | Platform where the record was created |
| `provider` | `string?` | LLM provider/service name |

**Serialized metadata (JSON strings):**
| Column | Type | Description |
|--------|------|-------------|
| `meta_json` | `string` | Additional record metadata |
| `llm_params_json` | `string` | LLM hyperparameters (model, temperature, etc.) |
| `llm_output_json` | `string` | Provider-specific output metadata (token counts, etc.) |

## Usage

```python
import datasets

ds = datasets.load_dataset("{HF_REPO_NAME}")

# access a specific split (train, valid, or test)
train_ds = ds["train"]

# filter by prompt type within a split
hints_ds = train_ds.filter(lambda row: row["prompt_name"] == "hints/docs")

# access a record
record = train_ds[0]
print(f"prompt name: {{record['prompt_name']}}")
print(f"response length: {{len(record['result'])}} chars")
```

## Source

Derived from the [PyINE-v1 trace dataset](https://huggingface.co/datasets/{SOURCE_DATASET_REPO})
using the [PyINE](https://github.com/saifh-github/pyine) framework (v{pyine.utils.reprod.get_framework_version()}).

**Providers queried:** {", ".join(f"`{p}`" for p in providers) if providers else "_not recorded_"}

**Models used:** {", ".join(f"`{m}`" for m in models) if models else "_not recorded_"}

"""

dataset_card = huggingface_hub.DatasetCard(card_content)
dataset_card.data = card_data
print(str(dataset_card))

In [ ]:
hf_dataset_dict.push_to_hub(
    repo_id=HF_REPO_NAME,
    max_shard_size=MAX_SHARD_SIZE,
)
dataset_card.push_to_hub(HF_REPO_NAME, repo_type="dataset")
print(f"pushed to https://huggingface.co/datasets/{HF_REPO_NAME}")